In [ ]:
from pathlib import Path
import sys

src_dir = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "common.py").is_file() and (p / "feature_preference").is_dir())
sys.path.insert(0, str(src_dir))


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms
from tqdm.auto import tqdm
import numpy as np
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import random

from common import *
import yaml
import matplotlib.colors as mcolors
import pickle

from hydra import initialize, initialize_config_module, initialize_config_dir, compose
from omegaconf import OmegaConf

import library

In [ ]:
device = torch.device("cuda")
sns.set_theme(style="whitegrid", font="cmr10", font_scale=1.9)
plt.rcParams["axes.formatter.use_mathtext"] = True
plt.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42})
colors = sns.color_palette("deep")
colors

default plot setting

In [ ]:
with initialize(version_base=None, config_path="configs"):
    config = compose(config_name="linear_barlowtwins_augmentation")

In [ ]:
batch = config.batch
sizes = config.sizes
s_in = sum(sizes)
s_out = len(sizes)

In [ ]:
x0, x1 = sample(batch, sizes, device, mul=0.01)
gamma, gamma_l, gamma_v = calculate_gamma(x0, x1)
sns.heatmap(gamma.cpu())
plt.show()
print(gamma_l, gamma_v)

In [ ]:
import os

crop_ratios = [1.0, 0.9, 0.8, 0.7, 0.6, 0.5]
print(f"Testing crop ratios: {crop_ratios}")

if os.path.exists(f"{config['network']}/data_augmentation.npy"):
    output_dict = np.load(f"{config['network']}/data_augmentation.npy", allow_pickle=True).item()
else:
    output_dict = {}

In [ ]:
def train(log_dict, encoder, lossFn, crop_ratio=None):
    set_seed(config.seed)
    print(encoder)
    epochs = int(config.epochs)
    lr = float(config.lr)
    scaling = float(config.scaling)

    optimizer = torch.optim.SGD(encoder.parameters(), lr, momentum=0, weight_decay=0)

    encoder.train()

    loss_values = []
    eiglist = [[] for i in range(s_out)]
    feature_alignment = [[] for _ in range(s_out)]
    weight_singular = [[] for _ in range(s_out)]
    subspace_alignment = []
    feature_weight_r_singular_alignment = [[[] for _ in range(2)] for _ in range(2)]

    if config.network.type != "linear":
        scale_weights(encoder, scaling ** (1 / config["layer"]))
    else:
        scale_weights(encoder, scaling)

    w = get_total_weight(encoder).detach().clone()
    u0, s0, v0T = torch.linalg.svd(w)
    v0 = v0T.T

    print(f"before {s0}")
    if config.network == "deep" and "BarlowTwins" in loss_type and config.balanced:
        with torch.no_grad():
            layer_s_value = s0 ** (1 / config["layer"])
            before_v = None
            modules = list(filter(lambda l: isinstance(l, nn.Linear), encoder.modules()))
            for index in reversed(range(len(modules))):
                module = modules[index]
                u, s, vt = torch.linalg.svd(module.weight.detach())
                s_next = torch.zeros(u.shape[0], vt.shape[0], device=u.device)
                for i in range(s.shape[0]):
                    s_next[i, i] = s[i]
                for i in range(layer_s_value.shape[0]):
                    s_next[i, i] = layer_s_value[i]
                if before_v is not None:
                    if index == 0 and config[loss_type]["aligned"]:
                        vt.T[:, :] = gamma_v[:, :].to(device)
                    module.weight.copy_(before_v @ s_next @ vt)
                else:
                    module.weight.copy_(u @ s_next @ vt)
                before_v = vt.T.clone()
            w = get_total_weight(encoder).detach().clone()
            u0, s0, v0T = torch.linalg.svd(w)
            v0 = v0T.T

            print(f"after {s0}")
    w = get_total_weight(encoder).detach().clone()
    u0, s0, v0T = torch.linalg.svd(w)
    print(f"after {s0}")
    v0 = v0T.T

    log_dict["w"] = w.cpu()
    log_dict["u0"] = u0.cpu()
    log_dict["s0"] = s0.cpu()
    log_dict["v0"] = v0.cpu()

    for i in range(s_out):
        weight_singular[i].append(s0[i].cpu().item())
        feature_alignment[i].append(torch.norm(e[i].unsqueeze(dim=0) @ w.T).cpu().item())
    subspace_alignment.append(
        (torch.norm(v0[:, :2].T @ gamma_v[:, :2].cuda(), p='fro') ** 2 / 2).cpu().item())
    print(subspace_alignment[0])

    for i in range(2):
        for j in range(2):
            feature_weight_r_singular_alignment[i][j].append(
                ((u0[:, i] @ w @ e[j].cuda()).cpu()).item())

    eig_reversed = False
    bar = tqdm(range(1, epochs + 1))
    u_prev = None

    for epoch in bar:
        optimizer.zero_grad()
        x0, x1 = sample(batch, sizes, device, mul=0.01)
        if crop_ratio is not None:
            x0, x1 = random_resized_crop_1d(x0, crop_ratio_min=crop_ratio, crop_ratio_max=1), random_resized_crop_1d(x1,
                                                                                                                     crop_ratio_min=crop_ratio,
                                                                                                                     crop_ratio_max=1)
        z0, z1 = torch.chunk(encoder(torch.cat([x0, x1], dim=0)), 2, dim=0)
        c, loss = lossFn(z0, z1)
        c = c.detach()
        v = reversed(torch.linalg.eigvalsh(c))
        for i in range(len(v)):
            # print(s0[0].item()**2 * gamma_l[0].item(), v[0].item())
            eiglist[i].append(v[i].cpu().item())
        loss_values.append(loss.item())
        loss.backward()
        optimizer.step()
        if epoch % 10 == 0:
            bar.set_description(f'loss: {loss.item():.4f} {v[0].item():.8f}  {v[1].item():.8f}')

        w_current = get_total_weight(encoder).detach().clone()
        u_w_current, s_w_current, v_w_current_h = torch.linalg.svd(w_current)
        v_w_current = v_w_current_h.T
        if epoch > 100:
            s_diff = abs(s_w_current[0] - s_w_current[1])
            if s_diff < 0.0005 and not eig_reversed:
                eig_reversed = True
                # u_w_current, s_w_current, v_w_current = align_svd_components(u_w_current, s_w_current, v_w_current, u_prev)
                # original_score = (abs(F.cosine_similarity(u_prev[0][:, 0], u_w_current[:, 0], dim=0)) + abs(F.cosine_similarity(u_prev[0][:, 1], u_w_current[:, 1], dim=0)))
                # flipped_score =  (abs(F.cosine_similarity(u_prev[0][:, 0], u_w_current[:, 1], dim=0)) + abs(F.cosine_similarity(u_prev[0][:, 1], u_w_current[:, 0], dim=0)))
                # eig_reversed = flipped_score > original_score
                # bar.set_description(f'loss: {loss.item():.4f} {original_score.item():.8f} {flipped_score.item():.8f}')
                # if eig_reversed:
                print("reversed")
        if eig_reversed:
            u_w_current = u_w_current[:, [1, 0]]
            s_w_current = s_w_current[[1, 0]]
            v_w_current = v_w_current[:, [1, 0]]
        if u_prev is None:
            u_prev = [u_w_current.clone()]
        if epoch > 100:
            u_prev.pop(0)
            u_prev.append(u_w_current.clone())

        # Measurement

        for i in range(s_out):
            weight_singular[i].append(s_w_current[i].cpu().item())
            feature_alignment[i].append(torch.norm(e[i].unsqueeze(dim=0) @ w_current.T).cpu().item())
        subspace_alignment.append(
            (torch.norm(v_w_current[:, :2].T @ gamma_v[:, :2].cuda(), p='fro') ** 2 / 2).cpu().item())

        for i in range(2):
            for j in range(2):
                feature_weight_r_singular_alignment[i][j].append(
                    ((u_w_current[:, i] @ w_current @ e[j].cuda()).cpu()).item())

    feature_weight_r_singular_alignment = np.array(feature_weight_r_singular_alignment)
    log_dict["loss"] = np.array(loss_values)
    log_dict["eiglist"] = np.array(eiglist)
    log_dict["weight_singular"] = np.array(weight_singular)
    log_dict["subspace_alignment"] = np.array(subspace_alignment)
    log_dict["feature_weight_r_singular_alignment"] = feature_weight_r_singular_alignment
    log_dict["feature_alignment"] = np.array(feature_alignment)

In [ ]:
e = create_segment_vectors(sizes).to(device)

method = config.method
lossFn = None
if method.loss == "BarlowTwins":
    lossFn = BTLoss(use_batchnorm=method.batch_norm, in_feature=s_out).to(device)
elif method.loss == "SimCLR":
    lossFn = SimCLRLoss(batch, 0.5).to(device)
else:
    lossFn = VICRegLoss().to(device)

network = config.network
if network.type == "linear":
    encoder = nn.Linear(s_in, s_out, bias=False, device=device)
elif network.type == "deep":
    mid_size = network.mid_layer_size
    total_layer = network.layer
    mid_layer = []
    for _ in range(total_layer - 2):
        mid_layer.append(nn.Linear(mid_size, mid_size, bias=False, device=device))
    encoder = nn.Sequential(nn.Linear(s_in, mid_size, bias=False, device=device), *mid_layer,
                            nn.Linear(mid_size, s_out, bias=False, device=device))
else:
    mid_size = network.mid_layer_size
    total_layer = network.layer
    mid_layer = []
    if network.activation == "leakyrelu" and "slope" in network:
        def activation_types():
            return nn.LeakyReLU(negative_slope=float(network.slope), inplace=True)
    else:
        def activation_types():
            return nn.ReLU(inplace=True)
    for _ in range(total_layer - 2):
        mid_layer.append(nn.Linear(mid_size, mid_size, bias=False, device=device))
        mid_layer.append(activation_types())
    encoder = nn.Sequential(nn.Linear(s_in, mid_size, bias=False, device=device), activation_types(), *mid_layer,
                            nn.Linear(mid_size, s_out, bias=False, device=device))

output_dict[method.loss] = {}
for crop_ratio in crop_ratios:
    output_dict[method.loss][crop_ratio] = {}
    lossFn.lambd = 1.0
    train(log_dict=output_dict[method.loss][crop_ratio], lossFn=lossFn, encoder=encoder, crop_ratio=crop_ratio)

In [ ]:
color_continuas = sns.color_palette("crest", n_colors=len(crop_ratios))
for (i, crop_ratio) in enumerate(crop_ratios):
    plt.plot(output_dict[method.loss][crop_ratio]["eiglist"][0], color=color_continuas[i], alpha=0.7,
             linestyle='--')
    plt.plot(output_dict[method.loss][crop_ratio]["eiglist"][1], color=color_continuas[i], alpha=0.7,
             label=f"{crop_ratio}")
plt.xlim(0, 4000)
plt.legend()

In [ ]:
set_seed(0)
sizes_override = [100, 200, 400]
batch_override = 1000
x0, x1 = sample(batch_override, sizes_override, device, mul=0)
gamma, gamma_l, gamma_v = calculate_gamma(x0, x1)
index = torch.argsort(gamma_l, descending=True)
gamma_l = gamma_l[index]
gamma_v = gamma_v[:, index]
print(gamma_l[0], gamma_l[1], gamma_l[2], "default")
plt.plot(x0[3, :].cpu(), color='red', alpha=0.5, linewidth=6.0, label=f'default', linestyle='-.')
for (i, crop_ratio) in enumerate(crop_ratios + [0.4, 0.3, 0.2, 0.1]):
    x0_crop = random_resized_crop_1d(x0, crop_ratio_min=crop_ratio, crop_ratio_max=crop_ratio)
    x1_crop = random_resized_crop_1d(x1, crop_ratio_min=crop_ratio, crop_ratio_max=crop_ratio)
    _, crop_gamma_l, crop_gamma_v = calculate_gamma(x0_crop, x1_crop)
    index = torch.argsort(crop_gamma_l, descending=True)
    crop_gamma_l = crop_gamma_l[index]
    crop_gamma_v = crop_gamma_v[:, index]
    # crop_gamma_l = sorted(crop_gamma_l, reverse=True)
    print(crop_gamma_l[0], crop_gamma_l[1], crop_gamma_l[2], crop_ratio)
    print(F.cosine_similarity(gamma_v.T, crop_gamma_v.T)[:4].abs())
    plt.plot(x0_crop[3, :].cpu(), color=color_continuas[i // 3], label=f'{crop_ratio}', alpha=0.8, linewidth=3.0)
plt.legend()

In [ ]:
for i in range(2):
    plt.plot(output_dict[loss_type]["eiglist"][i])

In [ ]:
np.save(f"{config["network"]}/data.npy", output_dict)

In [ ]:
def sample_with_center(batch, size, center_diff, total_size, device, mul=0.01):
    x_out = torch.zeros((batch, total_size), device=device)

    center = total_size // 2
    start = center + center_diff - size // 2
    end = start + size

    if start >= 0 and end <= total_size:
        rand = torch.rand((batch, 1), device=device).round() * 2 - 1
        x_out[:, start:end] = randv * rand.repeat(1, size)
        # x_out[:, start:end] = randv.repeat(batch, 1) * rand

    epsilon1 = mul * torch.randn_like(x_out)
    epsilon2 = mul * torch.randn_like(x_out)

    x1 = x_out + epsilon1
    x2 = x_out + epsilon2

    return x1, x2

batch = 100000
total_size = 256
total_list = []
total_list_others = []
total_vector = []
original_vector = []
size = 10
ratios = [0.3 + 0.1 * i for i in range(8)]
# randv = torch.rand((1, size), device=device) * 2 - 1
# randv = F.normalize(randv, dim=-1) * size
center_ratio = []
for ratio in tqdm(ratios):
    total_list.append([])
    total_list_others.append([])
    total_vector.append([])
    for i, center_diff in enumerate(range(0, total_size // 2 - size // 2, 5)):
        x0, x1 = sample_with_center(batch, size=size, center_diff=center_diff, total_size=total_size, device=device, mul=0.01)
        if ratio == 0.3:
            # original_vector.append(torch.round(x0[0]))
            original_vector.append(x0[0])
            center_ratio.append(center_diff / (total_size // 2) * 100.0)
        x0, x1 = random_resized_crop_1d(x0, ratio, ratio), random_resized_crop_1d(x1, ratio, ratio)
        gamma, gamma_l, gamma_v = calculate_gamma(x0, x1)
        total_list[-1].append(abs(gamma_l[0]))
        total_list_others[-1].append(abs(gamma_l[1:]))
        cossim = torch.cosine_similarity(original_vector[i].cpu().unsqueeze(dim=0), gamma_v[:, 0].unsqueeze(dim=0))
        total_vector[-1].append(abs(cossim))

print(gamma_l[:4])

In [ ]:
color_continuas = sns.color_palette("mako", n_colors=len(total_list))
fig, axs = plt.subplots(1, 1, figsize=(16, 8))
for i in range(len(ratios)):
    tl = np.array(total_list[i])
    # tl2 = np.array(total_list_others[i])
    axs.plot(center_ratio, tl, label=f'RRC ratio : {ratios[i]:.1f}', color=color_continuas[i], linewidth=3)
    # print(tl.shape, tl2.shape)
    # axs.plot(tl2[:, 0], color=color_continuas[i], linestyle='--', linewidth=3)
    # axs.plot(tl2[:, 8], color=color_continuas[i], linestyle='-.', linewidth=3)
# plt.legend()
axs.set_yscale('log')
axs.legend(loc='lower left', ncols=2)
axs.set_xlabel('Center Distance Percentage')
axs.set_ylabel('Maximum Eigenvalue')

# for i in range(len(ratios)):
#     axs[1].plot(total_vector[i], label=f'{ratios[i]:.1f}', color=color_continuas[i])

rand1 = torch.rand_like(original_vector[0]).cpu()*2-1
rand2 = torch.rand_like(rand1)*2-1
print(torch.cosine_similarity(rand1, rand2, dim=0))
# axs[1].axhline(abs(torch.cosine_similarity(rand1, rand2, dim=0)))

plt.tight_layout()
plt.savefig('rrc_center_distance.pdf')